In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# from google.colab import drive
# drive.mount('/content/drive')

In [11]:
import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [12]:
import json
import os
import re
import csv
import pandas as pd
from tqdm import tqdm
import gzip

In [13]:
def process_scrutin(scrutin: dict = None):
    all_votants = []
    
    main_data_keys = [
        "uid",
        "numero",
        "organeRef",
        "legislature",
        "seanceRef",
        "dateScrutin",
        "quantiemeJourSeance",
        "typeVote/codeTypeVote",
        "typeVote/typeMajorité",
        "sort/code",
        "titre",
        "objet/libelle",
        "syntheseVote/nombreVotants",
        "syntheseVote/suffragesExprimes",
        "syntheseVote/nbrSuffragesRequis",
        "syntheseVote/annonce",
        "syntheseVote/decompte/nonVotants",
        "syntheseVote/decompte/pour",
        "syntheseVote/decompte/contre",
        "syntheseVote/decompte/abstentions",
        "syntheseVote/decompte/nonVotantsVolontaires",
    ]

    main_data = {}
    for item in main_data_keys:
        parts = item.split("/")
        init_part = scrutin.get(parts[0])
        for key in parts[1:]:
            if init_part is None:
                break
            init_part = init_part.get(key)
        main_data[f"main/{item}"] = init_part

    ventilation = scrutin.get("ventilationVotes", {})
    
    # Handle both dict and list structures
    if isinstance(ventilation, dict):
        institutions = ventilation.values()
    elif isinstance(ventilation, list):
        institutions = ventilation
    else:
        return all_votants

    for institution in institutions:
        if institution is None:
            continue
            
        groupes = institution.get("groupes", {})
        groupe_list = groupes.get("groupe", [])
        
        # Handle both single group (dict) and multiple groups (list)
        if isinstance(groupe_list, dict):
            groupe_list = [groupe_list]

        for group in groupe_list:
            data_group_keys = [
                "organeRef",
                "nombreMembresGroupe",
                "vote/positionMajoritaire",
                "vote/decompteVoix/nonVotants",
                "vote/decompteVoix/pour",
                "vote/decompteVoix/contre",
                "vote/decompteVoix/abstentions",
                "vote/decompteVoix/nonVotantsVolontaires"
            ]

            group_data = {}
            for item in data_group_keys:
                parts = item.split("/")
                init_part = group.get(parts[0])
                for key in parts[1:]:
                    if init_part is None:
                        break
                    init_part = init_part.get(key)
                group_data[f"groupe/{item}"] = init_part

            decompte = group.get("vote", {}).get("decompteNominatif", {})
            
            for votant_type, votant_dict in decompte.items():
                if votant_dict is None:
                    continue
                if votant_dict == "0":
                    continue

                votant_list = votant_dict.get("votant")
                
                if votant_list is None:
                    continue

                # Handle both single votant (dict) and multiple votants (list)
                if isinstance(votant_list, dict):
                    votant_list = [votant_list]

                for votant in votant_list:
                    votant_payload = {}
                    votant_payload.update({f"votant/{k}": str(v) for k, v in votant.items()})
                    votant_payload.update({k: str(v) for k, v in group_data.items()})
                    votant_payload.update({k: str(v) for k, v in main_data.items()})
                    votant_payload.update({"votant/vote": votant_type})
                    all_votants.append(votant_payload)

    return all_votants

## XIVe Législature

In [14]:
# Cell: Load and process single large JSON file
import json

# Define paths
here = os.getcwd()
folder = os.path.dirname(here)
folder_0 = folder + "/0_raw"

# Path to the large JSON file
json_file_path = os.path.join(folder_0, "Scrutins_XIV.json", "Scrutins_XIV.json")

print(f"Loading JSON file: {json_file_path}...")

# Load the entire JSON file
with open(json_file_path, "r", encoding="UTF-8") as input_file:
    content = json.load(input_file)

print(f"JSON loaded successfully")
print(f"Top-level keys: {list(content.keys())[:10]}")

Loading JSON file: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Scrutins\Scrutins\1_data_extraction\14e_legislature_20juin2012-20juin2017/0_raw\Scrutins_XIV.json\Scrutins_XIV.json...
JSON loaded successfully
Top-level keys: ['scrutins']


In [ ]:
# Cell: Process JSON data with debugging
database = []

# Check the structure - adjust 'scrutins' key if different
scrutins_data = content.get("scrutins", content)

print(f"scrutins_data type: {type(scrutins_data)}")
print(f"scrutins_data keys (first 5): {list(scrutins_data.keys())[:5] if isinstance(scrutins_data, dict) else 'N/A'}")

# Handle both dict and list
if isinstance(scrutins_data, dict):
    scrutins_list = scrutins_data.get("scrutin", [])  
elif isinstance(scrutins_data, list):
    scrutins_list = scrutins_data
else:
    raise ValueError(f"Unexpected data structure: {type(scrutins_data)}")

print(f"Processing {len(scrutins_list)} scrutins...")
print(f"First scrutin type: {type(scrutins_list[0])}")

if len(scrutins_list) == 1 and isinstance(scrutins_list[0], list):
    print("Detected nested list structure, unwrapping...")
    scrutins_list = scrutins_list[0]

# Process each scrutin
error_count = 0
for idx, scrutin in enumerate(tqdm(scrutins_list)):
    if not isinstance(scrutin, dict):
        print(f"Scrutin {idx} is {type(scrutin)}, skipping")
        error_count += 1
        continue
        
    try:
        all_votants = process_scrutin(scrutin=scrutin)
        database.extend(all_votants)
    except Exception as e:
        print(f"Error processing scrutin {idx}: {e}")
        error_count += 1
        continue

print(f"\nTotal errors: {error_count}")
print(f"Total records extracted: {len(database)}")

df = pd.DataFrame.from_records(database)

# Ajouter la colonne manquante (par rapport aux autres législatures par des NaN
if 'votant/parDelegation' not in df.columns:
    df['votant/parDelegation'] = None

# Save to CSV only
output_folder = folder + "/1_extract_python"
output_path = os.path.join(output_folder, "scrutins.csv")
df.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Dataset shape: {df.shape}")

scrutins_data type: <class 'dict'>
scrutins_data keys (first 5): ['scrutin']
Processing 1354 scrutins...
First scrutin type: <class 'dict'>


100%|██████████| 1354/1354 [00:00<00:00, 2373.32it/s]



Total errors: 0
Total records extracted: 109913
Saved to: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Scrutins\Scrutins\1_data_extraction\14e_legislature_20juin2012-20juin2017/1_extract_python\scrutins.csv
Dataset shape: (109913, 33)


In [20]:
df.columns.tolist()

['votant/acteurRef',
 'votant/mandatRef',
 'votant/causePositionVote',
 'groupe/organeRef',
 'groupe/nombreMembresGroupe',
 'groupe/vote/positionMajoritaire',
 'groupe/vote/decompteVoix/nonVotants',
 'groupe/vote/decompteVoix/pour',
 'groupe/vote/decompteVoix/contre',
 'groupe/vote/decompteVoix/abstentions',
 'groupe/vote/decompteVoix/nonVotantsVolontaires',
 'main/uid',
 'main/numero',
 'main/organeRef',
 'main/legislature',
 'main/seanceRef',
 'main/dateScrutin',
 'main/quantiemeJourSeance',
 'main/typeVote/codeTypeVote',
 'main/typeVote/typeMajorité',
 'main/sort/code',
 'main/titre',
 'main/objet/libelle',
 'main/syntheseVote/nombreVotants',
 'main/syntheseVote/suffragesExprimes',
 'main/syntheseVote/nbrSuffragesRequis',
 'main/syntheseVote/annonce',
 'main/syntheseVote/decompte/nonVotants',
 'main/syntheseVote/decompte/pour',
 'main/syntheseVote/decompte/contre',
 'main/syntheseVote/decompte/abstentions',
 'main/syntheseVote/decompte/nonVotantsVolontaires',
 'votant/vote']